# Minima energía del hamiltoniano

## _Imports_

In [1]:
from scipy.optimize import minimize
from guppylang import guppy
from guppylang.defs import GuppyFunctionDefinition
from guppylang.std.builtins import comptime, result
from guppylang.std.angles import angle
from guppylang.std.quantum import h, measure, measure_array, qubit, ry, rx, cx, sdg, x, z

## Testeo de VQE

Según lo que creo, basta con cambiar el número de qubits `nq` y definir `anzats` correctamente para que el código aplique a nuestro problema. Y rezar un poco por que aún no lo he testeado.

In [2]:
nq = 2

@guppy
def anzats(qs: array[qubit, comptime(nq)], thetas: array[angle, comptime(nq)]) -> None:
    ry(qs[0], thetas[0])
    ry(qs[1], thetas[1])
    cx(qs[0], qs[1])

def make_circuit(thetas: list[float, nq], paulis: str) -> GuppyFunctionDefinition:
    mapping = {'I': 2, 'X': 0, 'Y': 1, 'Z': 2}
    code = [mapping.get(p, -1) for p in paulis]
    thetas = [float(th) for th in thetas]
    
    if -1 in code:
        raise ValueError(f"`paulis` must be in ['I', 'X', 'Y', 'Z'] and it was {paulis!r}")
    elif len(code) != nq:
        raise LengthError("`paulis` length must equal the number of qubits")
    
    @guppy
    def circuit() -> None:
        qs = array(qubit() for _ in range(comptime(nq)))
        anzats(qs, array(angle(th) for th in comptime(thetas)))
        
        i = 0
        for c in comptime(code):
            if c == 0:
                h(qs[i])
            elif c == 1:
                sdg(qs[i])
                h(qs[i])
            elif c == 2:
                pass
            else:
                pass
            i += 1
        
        result('m', measure_array(qs))
    return circuit

In [3]:
def expval(counts: Counter, paulis: str) -> float:
    total = sum(counts.values())
    expected_value = 0.0

    for measure, count in counts.items():
        ev = 1
        for i, pauli in enumerate(paulis):
            if pauli != 'I':
                ev *= (-1) ** int(measure[i])
        prob = count/total
        expected_value += ev * prob
    return expected_value

def energy(thetas: list[float, nq], hamiltonian: list[tuple[str, float]], n_shots = 4000) -> float:
    e = 0
    for pauli, coef in hamiltonian:
        circ = make_circuit(thetas, pauli)
        shots = circ.emulator(n_qubits=nq).with_shots(n_shots).run()
        counts = shots.register_counts()['m']
        e += coef * expval(counts, pauli)
    return e

In [4]:
def cost(thetas: list[float, nq]) -> float:
    hamiltonian = [
        ('ZI', 1.0),
        ('IZ', 1.0),
        ('XX', 1.0)
    ]
    return energy(thetas, hamiltonian, n_shots=4000)

In [5]:
result = minimize(cost, (0.1, 0.2), method='COBYLA')

In [6]:
result

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -2.1905
       x: [ 1.206e+00  1.008e-01]
    nfev: 30
   maxcv: 0.0